# 03. 분석 1 — 뇌 위축 예측모델 비교 (임상 블록)

연구기획서 4절과 4.1절을 실행한다. `code/tabular_model_comparison.py`의 파이프라인을 불러 **개발군 8,000명** 안에서 입력 블록(M0, M1, M2, M1−Dx)별로 여러 표형 모델을 같은 분할, 같은 전처리, 같은 튜닝 예산으로 비교한다. 유전체 블록(G, M1+G, M2+G, M1−Dx+G)은 노트북 04에서 APOE와 희소 PRS를 만든 뒤 이 노트북의 블록 목록에 추가해 같은 방식으로 실행한다.

- 실행 순서: 주 비교 블록(M1, M1+G, M1−Dx, M1−Dx+G)에서 전체 모델을 비교하고, 그 결과의 AUPRC 상위 2개 알고리즘만 나머지 블록(M0, M2, G, M2+G)에 적용한다(기획서 4.1절 프로토콜 5항).
- TabPFN: **TabPFN-3.5**를 버전 고정으로 쓴다(`tabpfn` 9.0.0 이상, 로컬 추론만; 원자료가 외부로 나가는 클라우드 API `tabpfn-client`는 쓰지 않는다). `USE_TABPFN = True`이면 포함하되, 라이선스 키(`TABPFN_TOKEN`)가 없거나 실행에 실패하면 자동으로 건너뛰고 나머지 모델로 계속한다. 라이선스 동의는 모델 버전마다 따로 필요하다(ux.priorlabs.ai, Licenses 탭). 맥에서 torch(TabPFN)와 LightGBM, XGBoost, CatBoost를 한 커널에서 함께 계산하면 커널이 죽을 수 있어, **TabPFN은 별도 프로세스(`code/tabpfn_worker.py`)에서 계산**하고 노트북 커널은 결과만 읽는다.
- `FAST = True`: 동작 확인용(외부 3분할 × 1회, 내부 2분할, 모델당 탐색 4회). 결과 해석에 쓰지 않는다.
- `FAST = False`: 기획서 프로토콜(외부 5분할 × 2회, 내부 3분할, 모델당 탐색 40회). 노트북 서버에서 실행하면 수십 분 걸린다.

평가군 2,000명은 이 노트북에서 쓰지 않는다. 주 모델을 사전 규칙으로 고른 뒤 노트북 05에서 한 번만 평가한다.

In [1]:
import sys, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))
import tabular_model_comparison as tmc

FAST = False                      # 동작 확인 후 False로 바꿔 정식 실행
USE_TABPFN = True                # TabPFN은 https://ux.priorlabs.ai 에서 라이선스 동의 후 받은 키를 TABPFN_TOKEN 환경변수에 넣어야 실행됨
import os
# TabPFN 키는 노트북에 적지 않고 환경변수 또는 code/.tabpfn_token 파일에서 읽는다(파일은 공유나 제출물에 포함하지 않는다)
_tok_file = ROOT / "code" / ".tabpfn_token"
if USE_TABPFN and not os.environ.get("TABPFN_TOKEN") and _tok_file.exists():
    os.environ["TABPFN_TOKEN"] = _tok_file.read_text(encoding="utf-8").strip()
if USE_TABPFN and not os.environ.get("TABPFN_TOKEN"):
    print("[알림] TABPFN_TOKEN이 없습니다. 키를 넣지 않으면 TabPFN은 건너뜁니다. (환경변수 또는 code/.tabpfn_token 파일)")
elif USE_TABPFN:
    print("TabPFN 키 로드됨 (길이 %d)" % len(os.environ["TABPFN_TOKEN"]))
MODELS = ["enet", "rf", "lgbm", "xgb", "cat"] + (["tabpfn"] if USE_TABPFN else [])   # 설치되지 않았거나 실패한 모델은 자동으로 건너뜀
import worker_util as wu          # TabPFN은 별도 프로세스에서 계산(커널에는 torch를 올리지 않음)
TABPFN_DEVICE, TABPFN_ENV = "auto", {}   # 0절 점검 결과로 바뀔 수 있음(auto 실패 시 cpu)
if USE_TABPFN:
    from importlib.metadata import version as _pkg_version, PackageNotFoundError
    try:
        print(f"tabpfn 패키지 {_pkg_version('tabpfn')} | 사용할 모델 버전 {tmc.Config().tabpfn_version} (9.0.0 이상 필요)")
    except PackageNotFoundError:
        print("[알림] tabpfn 패키지가 없습니다. %pip install -U \"tabpfn>=9.0.0\" 후 커널을 다시 시작하세요.")
DERIVED = ROOT / "05_분석결과" / "derived"; RES = ROOT / "05_분석결과" / "models"; RES.mkdir(parents=True, exist_ok=True)
B = pd.read_csv(DERIVED / "base_analysis.csv"); info = json.load(open(DERIVED / "normative_model.json", encoding="utf-8"))
C0, C1, C2 = info["blocks"]["C0"], info["blocks"]["C1"], info["blocks"]["C2"]
dev = B[B.split == "dev"].reset_index(drop=True)
print("개발군", len(dev), "| 위축 양성률 %.1f%%" % (100 * dev.atrophy_pos.mean()), "| FAST =", FAST)

TabPFN 키 로드됨 (길이 53)
tabpfn 패키지 9.0.0 | 사용할 모델 버전 v3.5 (9.0.0 이상 필요)
개발군 8000 | 위축 양성률 15.2% | FAST = False


## 0. TabPFN-3.5 점검 (별도 프로세스, 합성 난수)

실제 자료를 쓰지 않고 난수로 만든 표로 단계별 점검을 한다: torch 불러오기, 장치 연산, tabpfn 버전, 작은 적합(라이선스 확인과 가중치 내려받기), 본 실행 크기 적합(소요 시간 추정). 점검은 노트북 커널이 아닌 별도 프로세스에서 하므로, 문제가 있어도 커널은 죽지 않고 어느 단계에서 멈췄는지 보여 준다. 애플 GPU(MPS)로 실패하면 CPU로 한 번 더 시도하고, 통과한 장치를 본 실행에 쓴다.

In [2]:
TABPFN_DEVICE, TABPFN_ENV = None, {}
if USE_TABPFN:
    attempts = [("auto", {}), ("cpu", {})]
    for dev_opt, extra in attempts:
        print(f"\n----- 점검: 장치 {dev_opt}" + (f", 환경 {extra}" if extra else "") + " -----")
        rc, out = wu.run_worker(["check", "--device", dev_opt, "--version", tmc.Config().tabpfn_version], extra)
        if rc == 0:
            TABPFN_DEVICE, TABPFN_ENV = dev_opt, extra
            print(f"==> 통과: 본 실행은 장치 {dev_opt}로 계산합니다.")
            break
        why = wu.explain(rc, out); txt = "".join(out)
        print(f"==> 실패 (종료 코드 {rc}): {why}")
        if "라이선스" in why or "[실패] 모델 버전" in txt:
            break                                              # 장치를 바꿔도 해결되지 않는 문제
        if "OpenMP" in why and "KMP_DUPLICATE_LIB_OK" not in extra:
            attempts.append((dev_opt, {"KMP_DUPLICATE_LIB_OK": "TRUE"}))   # 마지막 수단(사용 시 결과표에 기록)
    if TABPFN_DEVICE is None:
        print("\nTabPFN 점검을 통과하지 못했습니다. 위 마지막 [단계] 줄과 실패 원인을 알려 주세요. 2절은 실행해도 TabPFN을 건너뜁니다.")
else:
    print("USE_TABPFN = False")


----- 점검: 장치 auto -----
[단계] 1. torch 불러오기
    torch 2.9.1 | MPS 사용 가능 True | CUDA False
[단계] 2. torch 기본 연산 (장치 auto)
    mps 행렬곱 합계 4672.2
[단계] 3. tabpfn 불러오기, 모델 버전 확인
    tabpfn 9.0.0 | 사용 가능 모델 ['v2', 'v2.5', 'v2.6', 'v3', 'v3.5', 'v3.5-fast']
[단계] 4. 작은 적합 200행 (라이선스 확인, 처음이면 가중치 내려받기)
    완료 11초 | 장치 (device(type='mps'),) | 모델 파일 tabpfn-v3.5-20260909.safetensors
[단계] 5. 본 실행 크기 적합 (훈련 6,400행, 예측 1,600행, 35열)
    완료 22초 → 주 비교 4블록 × 10폴드 예상 약 15분
{"ok": true, "device": "auto", "fold_seconds": 22.3, "total_seconds": 35.8}
==> 통과: 본 실행은 장치 auto로 계산합니다.


## 1. 입력 블록 정의

In [3]:
DZ = ["htn", "dm", "lip"]
C1_noDx = [c for c in C1 if c not in DZ]
BLOCKS = {"M0": C0, "M1": C0 + C1, "M2": C0 + C1 + C2, "M1-Dx": C0 + C1_noDx}
gpath = DERIVED / "genomic_features.csv"        # 노트북 04 산출물(있으면 자동 포함)
if gpath.exists():
    Gf = pd.read_csv(gpath); dev = dev.merge(Gf, on="T_ID", how="left")
    G = [c for c in Gf.columns if c != "T_ID"]
    BLOCKS.update({"G": C0 + G, "M1+G": C0 + C1 + G, "M2+G": C0 + C1 + C2 + G, "M1-Dx+G": C0 + C1_noDx + G})
    print("유전체 블록 포함:", G)
else:
    print("유전체 특징 파일이 아직 없어 임상 블록만 실행합니다.")
for k, v in BLOCKS.items(): print(f"  {k}: {len(v)}개 변수")

유전체 블록 포함: ['apoe_e4_count', 'apoe_e2_count', 'apoe_e4_carrier', 'prs_t2d', 'prs_htn', 'prs_lipid', 'prs_obesity', 'prs_alcohol', 'pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7', 'pc8', 'pc9', 'pc10']
  M0: 2개 변수
  M1: 17개 변수
  M2: 26개 변수
  M1-Dx: 14개 변수
  G: 20개 변수
  M1+G: 35개 변수
  M2+G: 44개 변수
  M1-Dx+G: 32개 변수


## 2. TabPFN만 추가 실행 (나머지 모델 결과 재사용)

나머지 5개 모델(Elastic-net, RF, LightGBM, XGBoost, CatBoost)의 개발군 OOF 예측은 이미 저장되어 있으므로, **TabPFN-3.5만 같은 분할(외부 5분할 × 2회, 같은 시드)로 계산**해 기존 결과에 합친다. 계산은 0절 점검을 통과한 장치로, 별도 프로세스에서 한다(진행 상황은 아래에 실시간 출력). 저장된 결과를 재사용할 수 없는 경우(설정, 변수, 대상자 순서 불일치)에는 5개 모델을 처음부터 다시 계산하지 않도록 멈춘다.

1. 주 비교 4블록(M1, M1+G, M1−Dx, M1−Dx+G)에서 TabPFN을 계산하고, 사전 규칙으로 주 모델을 다시 고른다.
2. M1+G의 AUPRC 상위 2개 알고리즘이 바뀌었는지 본다(기획서 4.1절 5항). 바뀌지 않았으면 나머지 블록(M0, M2, G, M2+G)은 그대로 두고, TabPFN이 상위 2개에 들어 바뀌었으면 기존 결과를 보관 폴더에 복사한 뒤 새 상위 2개로 나머지 블록을 이어서 계산한다(재사용 가능한 모델은 다시 계산하지 않음).
3. 블록별 요약표와 블록 간 증분 표(`block_increment.py`)를 갱신한다.

In [4]:
import shutil, runpy
prefix = "fast_" if FAST else "full_"
MAIN = [b for b in ["M1", "M1+G", "M1-Dx", "M1-Dx+G"] if b in BLOCKS]
REST = [b for b in BLOCKS if b not in MAIN]

def make_cfg(models):
    c = tmc.Config(task="classification", models=models)
    if FAST:
        c.outer_folds, c.repeats, c.inner_folds, c.n_iter, c.n_boot = 3, 1, 2, 4, 200
    return c

def resume_ok(name, cfg_):
    """저장된 결과를 재사용할 수 있는지 미리 확인한다(안 되면 전체 재계산이 일어나므로 멈춘다)."""
    d = RES / (prefix + name)
    if not ((d / "config.json").exists() and (d / "oof_predictions.csv").exists()):
        return False, "저장된 결과 없음"
    old = json.load(open(d / "config.json", encoding="utf-8"))
    for k in ["outer_folds", "repeats", "inner_folds", "n_iter", "seed"]:
        if old.get(k) != getattr(cfg_, k):
            return False, f"설정 불일치({k})"
    if old.get("features") != BLOCKS[name]:
        return False, "변수 목록 불일치"
    if pd.read_csv(d / "oof_predictions.csv", usecols=["T_ID"]).T_ID.astype(str).tolist() != dev.T_ID.astype(str).tolist():
        return False, "대상자 순서 불일치"
    return True, ""

def primary_of(name):
    p = RES / (prefix + name) / "config.json"
    return json.load(open(p, encoding="utf-8"))["primary_model"] if p.exists() else None

def has_oof(name, m, cfg_):
    d = RES / (prefix + name) / "oof_predictions.csv"
    if not d.exists(): return False
    cols = [f"{m}_rep{r}" for r in range(cfg_.repeats)]
    o = pd.read_csv(d, nrows=5)
    return all(c in o.columns for c in cols) and pd.read_csv(d, usecols=cols).notna().all().all()

def tabpfn_in_worker(name, models):
    """TabPFN만 별도 프로세스에서 계산하고, 나머지 모델은 저장된 OOF를 재사용해 결과 파일을 갱신한다."""
    spec = {"root": str(ROOT), "block": name, "features": BLOCKS[name], "models": models, "fast": FAST,
            "device": TABPFN_DEVICE, "tabpfn_version": tmc.Config().tabpfn_version, "out_dir": str(RES / (prefix + name))}
    rc, out = wu.run_spec("run", spec, TABPFN_ENV)
    if rc != 0:
        print(f"[실패] {name}: 종료 코드 {rc}, {wu.explain(rc, out)}")
    return rc == 0

before = {b: primary_of(b) for b in BLOCKS}
tab_sec = {}
if "tabpfn" not in MODELS:
    print("[알림] USE_TABPFN = False 입니다. 위에서 True로 바꾸고 다시 실행하세요.")
elif TABPFN_DEVICE is None:
    print("[중단] 0절 TabPFN 점검을 통과하지 못해 계산하지 않습니다.")
else:
    for name in MAIN:
        cfg_ = make_cfg(MODELS)
        ok, why = resume_ok(name, cfg_)
        if not ok:
            print(f"[중단] {name}: {why}. TabPFN만 따로 계산할 수 없습니다. 아래 전체 실행 셀(RUN_FULL = True)을 쓰세요.")
            continue
        t0 = time.time(); print(f"\n===== {name} ({len(BLOCKS[name])} 변수): TabPFN만 계산 (장치 {TABPFN_DEVICE}) =====")
        if not tabpfn_in_worker(name, MODELS):
            break
        tab_sec[name] = time.time() - t0

# TabPFN 결과만 모아 보기
rows = []
for name in MAIN:
    s = pd.read_csv(RES / (prefix + name) / "results_summary.csv").sort_values("AUPRC_mean", ascending=False).reset_index(drop=True)
    pw = pd.read_csv(RES / (prefix + name) / "pairwise_bootstrap.csv")
    r = {"블록": name}
    if "tabpfn" in set(s.model):
        t = s[s.model == "tabpfn"].iloc[0]; rank = int(s.index[s.model == "tabpfn"][0]) + 1
        d = pw[(pw.model == "tabpfn") & (pw.vs == "enet") & (pw.metric == "AUPRC")]
        r.update({"TabPFN AUPRC": round(t.AUPRC_mean, 4), "TabPFN AUROC": round(t.AUROC_mean, 4), "TabPFN Brier": round(t.Brier_mean, 4),
                  "TabPFN 보정 기울기": round(t.cal_slope_mean, 2), "TabPFN 민감도@20%": round(t["sens@20%_mean"], 3),
                  "AUPRC 순위": f"{rank}/{len(s)}",
                  "TabPFN − Elastic-net AUPRC (95% CI)": (f"{d['diff'].iloc[0]:+.4f} ({d.ci_lo.iloc[0]:+.4f}, {d.ci_hi.iloc[0]:+.4f})" if len(d) else "")})
    else:
        r.update({"TabPFN AUPRC": "실행 안 됨"})
    r.update({"기존 최고(AUPRC)": f"{s[s.model != 'tabpfn'].iloc[0].label} {s[s.model != 'tabpfn'].iloc[0].AUPRC_mean:.4f}",
              "주 모델 전 → 후": f"{before.get(name)} → {primary_of(name)}", "TabPFN 계산(분)": round(tab_sec.get(name, float('nan')) / 60, 1)})
    rows.append(r)
TT = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(f"\n[TabPFN-3.5 결과, 개발군 교차검증 | 장치 {TABPFN_DEVICE}" + (f", 환경 {TABPFN_ENV}" if TABPFN_ENV else "") + "]"); print(TT.to_string(index=False))
TT.to_csv(RES / (prefix + "tabpfn_only_summary.csv"), index=False, encoding="utf-8-sig")

# 상위 2개 알고리즘 변화 확인 → 필요할 때만 나머지 블록 갱신
ref = "M1+G" if "M1+G" in MAIN else "M1"
top2 = pd.read_csv(RES / (prefix + ref) / "results_summary.csv").sort_values("AUPRC_mean", ascending=False)["model"].head(2).tolist()
_p0 = RES / (prefix + REST[0]) / "config.json" if REST else None
prev = json.load(open(_p0, encoding="utf-8")).get("models") if _p0 is not None and _p0.exists() else None
print(f"\n{ref} AUPRC 상위 2개: 이전 {prev} → 현재 {top2}")
if prev is not None and sorted(prev) == sorted(top2):
    print("상위 2개가 그대로이므로 나머지 블록(" + ", ".join(REST) + ")은 다시 계산하지 않습니다.")
elif REST:
    arch = RES / ("_보관_이전상위2_" + "_".join(prev or ["없음"]))
    for b in REST:
        if (RES / (prefix + b)).exists():
            shutil.copytree(RES / (prefix + b), arch / (prefix + b), dirs_exist_ok=True)
    print(f"[보관] 기존 나머지 블록 결과를 {arch.name} 폴더에 복사했습니다(민감도 분석용). 새 상위 2개로 이어서 계산합니다.")
    others = [m for m in top2 if m != "tabpfn"]
    for name in REST:
        cfg_ = make_cfg(top2); ok, why = resume_ok(name, cfg_)
        if not ok:
            print(f"[중단] {name}: {why}."); continue
        t0 = time.time(); print(f"\n===== {name}: {top2} (재사용 가능한 모델은 불러옴) =====")
        if any(not has_oof(name, m, cfg_) for m in others) or "tabpfn" not in top2:
            # TabPFN이 아닌 모델은 커널에서 계산(또는 재사용), TabPFN은 여기서 계산하지 않음
            tmc.run(dev, "T_ID", "atrophy_pos", BLOCKS[name], [], None, cfg_, str(RES / (prefix + name)), resume=True, compute_only=others)
        if "tabpfn" in top2 and TABPFN_DEVICE is not None:
            tabpfn_in_worker(name, top2)
        print(f"블록 {name} 완료: {time.time() - t0:.0f}초")

print("\n주 모델(사전 규칙) 블록별, 전 → 후:")
for b in BLOCKS:
    a, z = before.get(b), primary_of(b)
    print(f"  {b:<8} {a} → {z}" + ("   (변경)" if a != z else ""))
if not FAST:
    print("\n블록 간 증분 표(표 2, 표 3) 갱신:")
    runpy.run_path(str(ROOT / "code" / "block_increment.py"))


===== M1 (17 변수): TabPFN만 계산 (장치 auto) =====
[단계] M1: TabPFN 계산 (장치 auto), 나머지 모델은 저장된 결과 재사용
[설정] task=classification  n=8000  p=17  models=['enet', 'rf', 'lgbm', 'xgb', 'cat', 'tabpfn']  outer=5x2  inner=3  n_iter=40
[이어서 실행] 기존 결과 재사용: ['enet', 'rf', 'lgbm', 'xgb', 'cat']
  TabPFN-3.5             repeat 0 fold 0  done (    19s)
  TabPFN-3.5             repeat 0 fold 1  done (    39s)
  TabPFN-3.5             repeat 0 fold 2  done (    59s)
  TabPFN-3.5             repeat 0 fold 3  done (    78s)
  TabPFN-3.5             repeat 0 fold 4  done (    99s)
  TabPFN-3.5             repeat 1 fold 0  done (   119s)
  TabPFN-3.5             repeat 1 fold 1  done (   139s)
  TabPFN-3.5             repeat 1 fold 2  done (   159s)
  TabPFN-3.5             repeat 1 fold 3  done (   180s)
  TabPFN-3.5             repeat 1 fold 4  done (   202s)

# 모델 비교 요약 (classification, n=8000, p=17)

외부 5분할 × 2회, 내부 3분할, 모델당 무작위 탐색 40회.
사전 고정 선택 규칙에 따른 **주 모델: Elastic-net (기준선)** (지표 AUPRC, 허용 오차 0.01).

| l

## 3. 전체 실행 (처음부터 계산하거나 전 블록을 다시 계산할 때만)

위 2절로 TabPFN을 추가했다면 이 셀은 실행하지 않아도 된다(`RUN_FULL = False`이면 건너뜀). `RUN_FULL = True`로 바꾸면 모든 블록을 실행하되, 같은 설정으로 이미 계산된 모델은 불러오고 빠진 모델만 계산한다.

In [5]:
RUN_FULL = False   # True: 전 블록 실행(저장된 모델은 재사용)
if not RUN_FULL:
    print("RUN_FULL = False: 전체 실행을 건너뜁니다(2절 결과 사용).")
else:
    cfg = tmc.Config(task="classification", models=MODELS)
    if FAST:
        cfg.outer_folds, cfg.repeats, cfg.inner_folds, cfg.n_iter, cfg.n_boot = 3, 1, 2, 4, 200
    prefix = "fast_" if FAST else "full_"
    summaries = {}
    FORCE_RERUN = False   # True면 저장된 결과가 있어도 전부 다시 계산

    def run_block(name, feats, cfg_):
        """이어서 실행: 같은 설정으로 이미 계산된 모델은 불러오고, 빠진 모델(예: 나중에 키를 넣은 TabPFN)만 계산해 합친다."""
        return tmc.run(dev, "T_ID", "atrophy_pos", feats, [], None, cfg_, str(RES / (prefix + name)), resume=not FORCE_RERUN,
                       compute_only=[m for m in cfg_.models if m != "tabpfn"])   # TabPFN은 커널에서 계산하지 않음(2절 사용)

    # 1단계: 주 비교 블록에서 전체 모델군 비교 (기획서 4.1절 프로토콜 5항)
    MAIN = [b for b in ["M1", "M1+G", "M1-Dx", "M1-Dx+G"] if b in BLOCKS]
    for name in MAIN:
        t0 = time.time(); print(f"\n===== 블록 {name} ({len(BLOCKS[name])} 변수) — 전체 모델 =====")
        summaries[name] = run_block(name, BLOCKS[name], cfg)
        print(f"블록 {name} 완료: {time.time()-t0:.0f}초")

    # 2단계: 주 비교 블록(M1+G, 없으면 M1)의 AUPRC 상위 2개 알고리즘만 나머지 블록에 적용
    ref = "M1+G" if "M1+G" in summaries else "M1"
    top2 = summaries[ref].sort_values("AUPRC_mean", ascending=False)["model"].head(2).tolist()
    print("\n나머지 블록에 적용할 상위 2개 알고리즘:", top2)
    # TabPFN이 나중에 추가되어 상위 2개가 바뀌면, 이전 상위 2개로 계산한 나머지 블록 결과를 덮어쓰기 전에 복사해 둔다(민감도 분석용)
    import shutil
    REST = [b for b in BLOCKS if b not in MAIN]
    _p0 = RES / (prefix + REST[0]) / "config.json" if REST else None
    _prev = json.load(open(_p0, encoding="utf-8")).get("models") if _p0 is not None and _p0.exists() else None
    if _prev and sorted(_prev) != sorted(top2):
        _arch = RES / ("_보관_이전상위2_" + "_".join(_prev))
        for b in REST:
            if (RES / (prefix + b)).exists():
                shutil.copytree(RES / (prefix + b), _arch / (prefix + b), dirs_exist_ok=True)
        print(f"[보관] 상위 2개가 {_prev} → {top2} 로 바뀌어 기존 나머지 블록 결과를 {_arch.name} 폴더에 복사했습니다.")
    cfg2 = tmc.Config(task="classification", models=top2)
    if FAST:
        cfg2.outer_folds, cfg2.repeats, cfg2.inner_folds, cfg2.n_iter, cfg2.n_boot = 3, 1, 2, 4, 200
    for name in [b for b in BLOCKS if b not in MAIN]:
        t0 = time.time(); print(f"\n===== 블록 {name} ({len(BLOCKS[name])} 변수) — {top2} =====")
        summaries[name] = run_block(name, BLOCKS[name], cfg2)
        print(f"블록 {name} 완료: {time.time()-t0:.0f}초")

RUN_FULL = False: 전체 실행을 건너뜁니다(2절 결과 사용).


## 4. 블록 × 모델 요약표

In [6]:
prefix = "fast_" if FAST else "full_"
summaries = {b: pd.read_csv(RES / (prefix + b) / "results_summary.csv") for b in BLOCKS if (RES / (prefix + b) / "results_summary.csv").exists()}
rows = []
for name, s in summaries.items():
    for _, r in s.iterrows():
        rows.append({"블록": name, "모델": r["label"], "AUPRC": round(r["AUPRC_mean"], 3), "AUROC": round(r["AUROC_mean"], 3),
                     "Brier": round(r["Brier_mean"], 4), "보정 기울기": round(r["cal_slope_mean"], 2),
                     "민감도@20%": round(r["sens@20%_mean"], 3), "MRI/1명@20%": round(r["MRI_per_case@20%_mean"], 2), "초": r["train_seconds"]})
T = pd.DataFrame(rows)
T.to_csv(RES / (prefix + "summary_all_blocks.csv"), index=False, encoding="utf-8-sig")
print(T.to_string(index=False))
print("\n주 모델(사전 규칙) 블록별:")
for name in summaries:
    cfgj = json.load(open(RES / (prefix + name) / "config.json", encoding="utf-8"))
    print(f"  {name}: {cfgj['primary_model']}")

     블록                모델  AUPRC  AUROC  Brier  보정 기울기  민감도@20%  MRI/1명@20%      초
     M0          CatBoost  0.166  0.527 0.1288    0.45    0.217        6.08   51.7
     M0           XGBoost  0.160  0.512 0.1292    0.22    0.208        6.34    7.5
     M1 Elastic-net (기준선)  0.239  0.629 0.1243    1.14    0.357        3.69   35.5
     M1        TabPFN-3.5  0.235  0.627 0.1244    0.96    0.353        3.73  202.3
     M1          CatBoost  0.233  0.624 0.1248    0.92    0.353        3.74  825.2
     M1     Random Forest  0.231  0.619 0.1252    1.42    0.354        3.73  433.4
     M1           XGBoost  0.228  0.619 0.1255    0.80    0.350        3.77 1121.8
     M1          LightGBM  0.220  0.609 0.1269    0.63    0.332        3.97 5009.2
     M2          CatBoost  0.234  0.625 0.1248    0.95    0.353        3.73  176.2
     M2           XGBoost  0.227  0.617 0.1256    0.81    0.345        3.82   28.0
  M1-Dx     Random Forest  0.180  0.546 0.1282    0.85    0.240        5.49  430.9
  M1